<a href="https://colab.research.google.com/github/dvalverdes/prueba_tecnica_DanielValverdeSalazar/blob/main/bloque2_decisiones.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bloque 2 - Modelo Dimensional + Diseño de Pipeline

## A. Modelo Dimensional

### Objetivo
Diseñar un **Star Schema en BigQuery** que soporte los siguientes análisis del negocio:

- Comp Sales por tienda / formato / país / período
- GMROI por vendor / categoría / región
- Retención de clientes por cohorte
- Productividad de tienda
- Análisis de promociones

---

## 1. Tablas de hechos y dimensiones con sus campos clave

### Tabla de hechos principal: `fact_sales_item`
**Grano:** una fila por `transaction_id + item_id`.

**Propósito:** almacenar el detalle de ventas a nivel ítem, permitiendo análisis de ventas, promociones, rentabilidad, cohortes, productividad y basket analysis.

**Campos clave:**
- `sales_item_key` (PK)
- `date_key` (FK)
- `store_key` (FK)
- `product_key` (FK)
- `vendor_key` (FK)
- `customer_key` (FK, nullable)
- `transaction_id` (degenerate dimension)
- `transaction_item_id` (degenerate dimension)

**Métricas principales:**
- `quantity`
- `unit_price`
- `sales_amount`
- `cost_amount`
- `gross_margin`
- `promotion_flag`
- `loyalty_card_flag`

---

### Tabla de hechos secundaria: `fact_store_promotion`
**Grano:** una fila por promoción asignada a una tienda en un rango de fechas.

**Propósito:** modelar promociones y experimentos por tienda, incluyendo variantes de A/B test y tipo de promoción.

**Campos clave:**
- `store_promo_key` (PK)
- `store_key` (FK)
- `start_date_key` (FK)
- `end_date_key` (FK)

**Atributos principales:**
- `promo_name`
- `variant`
- `promo_type`

---

## 2. Dimensiones

### `dim_date`
**Propósito:** permitir análisis temporales consistentes.

**Campos clave:**
- `date_key` (PK)

**Atributos principales:**
- `full_date`
- `day`
- `month`
- `month_name`
- `quarter`
- `year`
- `week`
- `year_month`
- `day_of_week`

---

### `dim_store`
**Propósito:** describir el contexto de la tienda.

**Campos clave:**
- `store_key` (PK)
- `store_id` (NK)

**Atributos principales:**
- `store_name`
- `country`
- `city`
- `format`
- `size_sqm`
- `opening_date`
- `region`

---

### `dim_product`
**Propósito:** describir el producto vendido.

**Campos clave:**
- `product_key` (PK)
- `item_id` (NK)
- `vendor_key` (FK)

**Atributos principales:**
- `item_name`
- `brand`
- `category`
- `department`
- `cost`
- `is_shared_catalog`

---

### `dim_vendor`
**Propósito:** describir al proveedor asociado al producto.

**Campos clave:**
- `vendor_key` (PK)
- `vendor_id` (NK)

**Atributos principales:**
- `vendor_name`
- `country`
- `tier`

---

### `dim_customer`
**Propósito:** describir al cliente identificado cuando exista información de lealtad.

**Campos clave:**
- `customer_key` (PK)
- `customer_id` (NK, nullable)

**Atributos principales:**
- `loyalty_flag`
- `first_purchase_date`
- `customer_type`

---

## 3. Justificación de decisiones de diseño

### Decisión 1. Definir el grano de la tabla principal a nivel `transaction_id + item_id`
Se eligió este grano porque los análisis requeridos dependen del detalle de producto y no solo del ticket total. Por ejemplo:

- promociones (`was_on_promo`) se observan a nivel ítem
- GMROI requiere costo y margen por producto
- posibles quiebres de stock se detectan por tienda-item-fecha
- basket analysis necesita detalle por categoría e ítem

Si el modelo se hubiera diseñado solo a nivel transacción, se perdería el detalle necesario para responder varios casos del negocio.

---

### Decisión 2. Modelar `customer_id` como campo opcional en `dim_customer`
Aproximadamente el 60% de las transacciones no tiene `customer_id`, porque corresponden a compras sin identificación de cliente o sin uso de tarjeta de lealtad. Por esa razón, no se forzó una relación obligatoria con cliente.

La solución fue:
- permitir `customer_key` nulo en la tabla de hechos, o
- mapear esos casos a un registro especial tipo `UNKNOWN_CUSTOMER`

Esta decisión evita perder transacciones válidas y al mismo tiempo permite analizar correctamente cohortes y retención solo para clientes identificados.

---

### Decisión 3. Mantener `transaction_id` y `transaction_item_id` como dimensiones degeneradas
Se decidió no crear una dimensión separada de transacción porque esos campos no aportan atributos descriptivos relevantes, pero sí son necesarios para:

- trazabilidad
- auditoría
- validación de duplicados
- reconstrucción del ticket
- análisis a nivel transacción

Por eso se almacenan directamente en la tabla de hechos como dimensiones degeneradas.

---

### Decisión 4. Separar promociones en una tabla de hechos independiente
Las promociones no son solo un atributo fijo del producto, porque tienen comportamiento temporal y además están asignadas a nivel tienda.

Una promoción puede variar por:
- tienda
- rango de fechas
- nombre del experimento
- variante (`CONTROL` / `TREATMENT`)
- tipo de promoción

Por eso se creó `fact_store_promotion` como un hecho separado. Esto permite soportar correctamente análisis de promociones y A/B testing sin contaminar la fact principal de ventas.

---

### Decisión 5. Incorporar una dimensión de fecha (`dim_date`)
Se incluyó `dim_date` para simplificar y estandarizar los análisis temporales. Esto es importante porque los requerimientos incluyen:

- Comp Sales año contra año
- cohortes mensuales
- productividad del último trimestre
- análisis de gaps de fechas
- promociones con rango temporal

La dimensión fecha permite manejar estos análisis de forma consistente y mejora la legibilidad del modelo en BigQuery.

---

## 4. Cómo este modelo soporta los requerimientos del negocio

Este modelo dimensional soporta los análisis solicitados de la siguiente manera:

- **Comp Sales:** mediante `fact_sales_item` + `dim_store` + `dim_date`
- **GMROI por vendor/categoría/región:** mediante `fact_sales_item` + `dim_product` + `dim_vendor` + `dim_store`
- **Retención por cohortes:** mediante `fact_sales_item` + `dim_customer` + `dim_date`
- **Productividad de tienda:** mediante `fact_sales_item` + `dim_store`
- **Promociones y A/B test:** mediante `fact_store_promotion` + `fact_sales_item` + `dim_store` + `dim_date`

---

## 5. Conclusión
El modelo propuesto se centra en una tabla de hechos de ventas a nivel ítem, complementada con dimensiones descriptivas y una tabla de hechos específica para promociones. Esta estructura permite responder de manera flexible y escalable a los requerimientos analíticos del negocio, manteniendo un diseño claro, consistente y adecuado para un entorno analítico como BigQuery.

```markdown
## B. Diseño del Pipeline ETL/ELT

### Objetivo
Diseñar un pipeline de datos que permita cargar, transformar y disponibilizar información de ventas de forma confiable en BigQuery, considerando retrasos operativos, cargas incrementales, detección de fallas y necesidades de refresh diario para dashboards.

---

## 1. Manejo de tiendas que reportan ventas con hasta 2 horas de retraso

Para manejar retrasos de hasta 2 horas en el envío de datos por parte de las tiendas, propondría un pipeline con una estrategia de **watermark temporal** y una **ventana de re-procesamiento**.

### Propuesta
- Definir una capa **raw/staging** donde se almacenen todos los archivos o eventos recibidos sin transformar.
- Registrar para cada carga:
  - fecha/hora de ingestión
  - fecha/hora de transacción
  - tienda origen
  - identificador de lote o archivo
- Configurar el pipeline para reprocesar siempre una ventana reciente, por ejemplo las últimas **4 a 6 horas**, en lugar de cargar solo datos estrictamente nuevos.
- Utilizar `transaction_id` y `transaction_item_id` como claves de deduplicación para evitar duplicados al reprocesar.

### Justificación
Si una tienda envía datos con hasta 2 horas de retraso, una carga estrictamente incremental podría dejar transacciones fuera del modelo analítico. Reprocesar una ventana reciente permite capturar llegadas tardías sin perder información.

### Decisión
Usar una estrategia de **incremental load + lookback window**, de modo que el pipeline tolere retrasos operativos sin sacrificar consistencia.

---

## 2. Detección automática de una tienda que dejó de enviar datos

Para detectar automáticamente si una tienda dejó de enviar información, propondría combinar **controles de frescura** con **alertas operativas**.

### Propuesta
- Mantener una tabla de monitoreo por tienda con:
  - `store_id`
  - última fecha/hora recibida
  - cantidad de transacciones del día
  - cantidad de archivos/lotes recibidos
  - estado del pipeline
- Definir una regla de negocio basada en expectativa operativa, por ejemplo:
  - si una tienda no reporta datos dentro de un umbral esperado (por ejemplo, más de 3 o 4 horas sin actividad dentro de horario operativo), generar alerta
- Complementar con validaciones históricas:
  - comparar volumen actual vs promedio histórico por tienda y franja horaria
  - identificar caídas abruptas a cero que no sean normales

### Ejemplo de alerta
- **Alerta amarilla:** tienda sin datos en las últimas 2 horas
- **Alerta roja:** tienda sin datos en las últimas 4 horas o sin transacciones durante todo el día

### Decisión
Implementar controles de **freshness SLA** por tienda y un tablero de monitoreo operativo para detectar interrupciones de envío de datos antes de que impacten el dashboard final.

---

## 3. Cargas incrementales sin duplicar transacciones

Para hacer cargas incrementales sin duplicar datos, usaría una estrategia de **upsert / merge** basada en claves de negocio.

### Claves recomendadas
- A nivel transacción: `transaction_id`
- A nivel ítem: `transaction_item_id`
- En promociones: combinación de `store_id + promo_name + start_date + end_date + variant`

### Propuesta
1. Cargar datos nuevos en una tabla staging.
2. Ejecutar validaciones de calidad:
   - duplicados
   - integridad referencial
   - nulos críticos
3. Usar un `MERGE` hacia las tablas destino:
   - si la clave ya existe, actualizar
   - si no existe, insertar
4. Reprocesar una ventana reciente para capturar late arrivals.
5. Mantener metadata de ejecución:
   - timestamp de carga
   - watermark procesado
   - cantidad de inserts / updates

### Justificación
Las cargas incrementales deben ser **idempotentes**, es decir, si el pipeline se ejecuta más de una vez sobre el mismo rango de datos, el resultado final no debe duplicar registros.

### Decisión
Usar patrón **staging + MERGE + deduplicación por claves de negocio**, apoyado por una ventana de reproceso para tolerar datos tardíos.

---

## 4. Frecuencia del pipeline si el dashboard requiere refresh diario

Si el dashboard necesita refresh diario, no correría el pipeline solo una vez al día. Lo ideal sería separar:

- **frecuencia de ingestión**
- **frecuencia de publicación al dashboard**

### Propuesta
- Ejecutar ingestión y transformaciones incrementales varias veces al día, por ejemplo:
  - cada 1 hora
  - o cada 30 minutos si el volumen lo permite
- Publicar una versión consolidada para reporting al menos una vez al día, por ejemplo:
  - cierre operativo nocturno
  - o actualización final temprana en la mañana

### Enfoque recomendado
- **Raw/Staging:** cargas frecuentes
- **Curated/Analytics:** actualización programada con lógica de consolidación
- **Dashboard:** refresh diario sobre tablas ya curadas

### Justificación
Aunque el negocio pida un dashboard diario, correr el pipeline más frecuentemente mejora:
- monitoreo operativo
- detección temprana de fallas
- manejo de retrasos
- menor acumulación de errores al cierre

### Decisión
Configurar el pipeline incremental con frecuencia horaria y publicar una capa analítica consolidada diaria para consumo de dashboards.

---

## 5. Propuesta de arquitectura lógica del pipeline

### Capas sugeridas

#### 1. Raw
- Recepción de archivos o eventos originales
- Sin transformaciones
- Conserva trazabilidad completa

#### 2. Staging
- Normalización de tipos
- tratamiento de nulos
- deduplicación inicial
- validaciones básicas

#### 3. Curated / Silver
- joins con catálogos
- enriquecimiento
- cálculo de métricas base
- aplicación de reglas de negocio

#### 4. Data Mart / Gold
- tablas dimensionales
- hechos analíticos
- datasets listos para reporting

### Beneficios
- mejor trazabilidad
- separación clara entre ingestión y analítica
- facilidad para re-procesar
- control de calidad por capa

---

## 6. Conclusión
El pipeline propuesto está diseñado para ser confiable, tolerante a retrasos y fácil de operar. La estrategia combina:

- ingestión incremental con ventana de reproceso
- deduplicación por claves de negocio
- monitoreo de frescura por tienda
- publicación diaria sobre una capa analítica consolidada

Con este enfoque se puede soportar un dashboard diario sin perder robustez operativa ni calidad de datos.
```


## C. Gobernanza de Datos

### Objetivo
Definir lineamientos de gobernanza para asegurar que los datos analíticos sean confiables, trazables, seguros y adecuados para uso de negocio, especialmente en un contexto de retail con información transaccional, promociones y clientes identificados por lealtad.

---

## 1. Controles de calidad que implementaría

La gobernanza debe comenzar con controles automáticos de calidad en cada etapa del pipeline. No basta con cargar datos; es necesario validar que cumplan reglas mínimas antes de ser publicados en la capa analítica.

### Controles propuestos

#### Completitud
Validar presencia de campos críticos:
- `transaction_id`
- `transaction_date`
- `store_id`
- `item_id`
- `quantity`
- `unit_price`

También controlar nulos permitidos vs no permitidos, por ejemplo:
- `customer_id` puede venir nulo
- `transaction_id` no debería venir nulo

#### Unicidad
Verificar duplicados en claves de negocio:
- `transaction_id`
- `transaction_item_id`
- combinaciones esperadas en promociones

#### Validez
Revisar reglas como:
- `quantity > 0`
- `unit_price >= 0`
- `total_amount >= 0` salvo casos excepcionales documentados
- `variant` ∈ {`CONTROL`, `TREATMENT`}
- formatos de fecha correctos

#### Integridad referencial
Validar que:
- `store_id` exista en `stores`
- `item_id` exista en `products`
- `vendor_id` exista en `vendors`

#### Frescura
Monitorear:
- última fecha/hora recibida por tienda
- retraso entre fecha de transacción y fecha de ingestión
- gaps anómalos de actividad

#### Consistencia
Comparar:
- `total_amount` vs suma de `unit_price * quantity`
- promociones activas vs flags transaccionales
- coherencia entre `loyalty_card` y `customer_id`

### Decisión
Implementaría estos controles como validaciones automáticas en staging y curated, con resultados registrados en una tabla de auditoría de calidad.

---

## 2. Política de PII y tratamiento de `customer_id`

El campo `customer_id` debe considerarse información sensible o cuasi sensible, ya que permite identificar o perfilar clientes, especialmente al combinarse con comportamiento de compra.

### Principios de manejo

#### Minimización
Solo usar `customer_id` cuando sea necesario para análisis como:
- cohortes
- retención
- frecuencia de compra
- comportamiento loyalty

No exponerlo innecesariamente en dashboards ejecutivos.

#### Pseudonimización
Transformar `customer_id` en un identificador hasheado o tokenizado para capas analíticas intermedias y de consumo.

Ejemplo:
- en raw puede existir el identificador original
- en silver/gold usar `customer_id_hash`

#### Acceso restringido
Permitir acceso al identificador original solo a perfiles autorizados:
- Data Engineering
- Data Governance
- ciertos analistas aprobados

#### Separación por capas
- **Raw:** puede contener el identificador original bajo controles estrictos
- **Curated / Gold:** preferiblemente usar versión hasheada o surrogate key

#### Retención controlada
No mantener el identificador original más tiempo del necesario si el negocio no lo requiere.

### Decisión
Tratar `customer_id` como dato sensible de cliente, aplicando pseudonimización, acceso restringido y principio de mínimo privilegio.

---

## 3. Definición de ownership de métricas

Uno de los riesgos más comunes en analítica es que distintas áreas calculen la misma métrica de formas diferentes. Para evitarlo, cada métrica crítica debe tener un dueño de negocio y una definición única.

### Métricas que deberían tener ownership explícito
- GMV
- Comp Sales Growth %
- GMROI
- Ticket promedio
- Retención
- Productividad por m²
- GMV perdido estimado por quiebre
- Métricas de promo uplift

### Propuesta de ownership

#### Negocio / Comercial
Dueño conceptual de métricas como:
- GMV
- ticket promedio
- comp sales
- promo uplift

#### Finanzas / Control de gestión
Dueño conceptual de métricas como:
- margen bruto
- GMROI
- costo total

#### Data / BI
Dueño técnico de:
- implementación de la lógica
- consistencia entre dashboards
- versionado de definiciones
- documentación técnica

### Qué documentaría para cada métrica
- nombre oficial
- definición de negocio
- fórmula
- granularidad
- fuente de datos
- filtros / exclusiones
- owner de negocio
- owner técnico

### Decisión
Crear un catálogo de métricas con ownership dual:
- **owner de negocio**
- **owner técnico**

Así se evita ambigüedad y se mejora la confianza en los reportes.

---

## 4. Estrategia de particionado en BigQuery

El particionado es clave para rendimiento, costo y mantenimiento en BigQuery.

### Recomendación principal
Particionar la tabla principal `fact_sales_item` por fecha de transacción.

#### Campo recomendado
- `transaction_date`
o su equivalente modelado en la fact:
- `date_key` / `full_date`

### Justificación
La mayoría de los análisis del negocio filtran por tiempo:
- ventas del trimestre
- comp sales YoY
- cohortes mensuales
- gaps de fechas
- desempeño de promociones por período

Particionar por fecha permite:
- escanear menos datos
- reducir costo
- acelerar consultas

### Clustering recomendado
Además del particionado, usar clustering por columnas de alta selectividad analítica, por ejemplo:
- `store_key`
- `product_key`
- `category`
- `vendor_key`

### En promociones
`fact_store_promotion` podría particionarse por:
- `start_date_key`
o por fecha efectiva de promoción

### Decisión
Particionar `fact_sales_item` por fecha de transacción y complementar con clustering por tienda y producto, ya que esta combinación optimiza los principales patrones de consulta del negocio.

---

## 5. Linaje, trazabilidad y auditoría

La gobernanza también requiere poder explicar de dónde vino cada dato y cómo fue transformado.

### Elementos que incluiría
- tabla de control de ejecuciones del pipeline
- timestamp de ingestión
- nombre del archivo o lote origen
- watermark procesado
- cantidad de filas leídas, insertadas, actualizadas, rechazadas
- resultado de validaciones de calidad

### Beneficio
Esto permite:
- investigar incidentes
- re-procesar lotes
- explicar discrepancias
- auditar cambios en métricas

### Decisión
Mantener metadatos de linaje y auditoría como parte obligatoria del pipeline.

---

## 6. Acceso y seguridad

La gobernanza también implica definir quién puede ver qué.

### Propuesta
Aplicar control de acceso por capas:

#### Raw
Acceso muy restringido, porque puede contener:
- identificadores originales
- datos sin depurar
- información sensible

#### Curated / Silver
Acceso controlado para ingeniería y analistas autorizados.

#### Gold / Data Mart
Acceso orientado a consumo de negocio, con datos ya estandarizados y con PII minimizada.

### Principio recomendado
Usar **least privilege**:
cada rol accede solo a lo que necesita.

### Decisión
Separar permisos por capa y limitar acceso a PII y datos crudos.

---

## 7. Conclusión
La propuesta de gobernanza busca asegurar que los datos sean no solo útiles, sino también confiables, seguros y sostenibles en el tiempo. Para ello se plantea:

- controles automáticos de calidad
- tratamiento seguro de `customer_id`
- ownership claro de métricas
- particionado eficiente en BigQuery
- trazabilidad end-to-end
- seguridad por capas y por rol

Con este enfoque, la solución analítica no solo responde preguntas de negocio, sino que también puede operar con estándares adecuados de calidad, desempeño y control.